In [8]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl
import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC") 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import datetime

from BT.data_handler import TimeGrid
from BT.misc import ql_cal_date_range
from BT.query_actions import AddQueryAction, UnwindPositionsAction
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy
from BT.triggers import DateTrigger, DateTriggerRequirements

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapStructure import IRSwapStructure
from Query.IRSwaps.IRSwapValue import IRSwapValue

In [10]:
# mdp = IRSwapsMDP(source="GSQUANT-rl_basic")
mdp = IRSwapsMDP(source="CME_NY_EOD_LIVE-ql_basic")

In [11]:
curve = "USD-FEDFUNDS"
# curve_handle = mdp._get_curve(curve_name=curve, timestamp=datetime.date(2025, 10, 1), kwargs={"force_refresh": True})
curve_handle = mdp._get_curve(curve_name=curve, timestamp=datetime.date(2025, 10, 1))

In [12]:
risk = 100_000
outright_query = IRSwapQuery(curve=curve, tenor="10Y", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": risk})
outright_pkg, outright_rws = outright_query.resolve_package(pricer_or_curve=curve_handle) 

outright_vmap = outright_query.build_value_map(pricer_or_curve=curve_handle, package=outright_pkg, risk_weights=outright_rws)
for v in [IRSwapValue.RATE, IRSwapValue.NPV, IRSwapValue.NOTIONAL, IRSwapValue.PV01]:
    print(v.name, outright_vmap.apply(value=v))

RATE 3.5587918794827402
NPV 2.9802322387695312e-08
NOTIONAL 118375501.30741017
PV01 100000.0


In [13]:
outright_vmap.apply(value=IRSwapValue.ROLL_BPS_RUNNING, **{"horizon": "6m"})

2.2660615845315757

In [20]:
def _normalize_leg(s: str) -> str:
    import re
    # grab tenor tokens like 3M, 6m, 1Y, 2y (case-insensitive)
    parts = re.findall(r'\d+\s*[dwmy]', s, flags=re.I)
    parts = [p.upper().replace(" ", "") for p in parts]
    if len(parts) == 1:
        return parts[0]
    if len(parts) == 2:
        return f"{parts[0]}x{parts[1]}"
    return s.split("/")


fly_str = "2y/5y/10y" 
wing1, belly, wing2 = _normalize_leg(fly_str) 

risk = -100_000
fly_query = IRSwapQuery(curve=curve, structure=IRSwapStructure.FLY, structure_kwargs={"front_tenor": wing1, "belly_tenor": belly, "back_tenor": wing2, "bpv": risk})
fly_pkg, fly_rws = fly_query.resolve_package(pricer_or_curve=curve_handle)

fly_vmap = fly_query.build_value_map(pricer_or_curve=curve_handle, package=fly_pkg, risk_weights=fly_rws)
print(fly_str, fly_vmap.apply(value=IRSwapValue.CARRY_BPS_RUNNING, **{"horizon": "3m"}))

2y/5y/10y -0.4573870928885331


In [15]:
fomc_query = IRSwapQuery(curve=curve, tenor="oct25", structure_kwargs={"bpv": 100_000})
fomc_pkg, fomc_rws = fomc_query.resolve_package(pricer_or_curve=curve_handle)

fomc_vmap = fomc_query.build_value_map(pricer_or_curve=curve_handle, package=fomc_pkg, risk_weights=fomc_rws)
fomc_vmap.apply(value=IRSwapValue.RATE)

3.855651701355013

In [16]:
fomc_pkg[0].cashflows()

AttributeError: 'OvernightIndexedSwap' object has no attribute 'cashflows'